# PredictLab — Notebook 01: Data Quality Audit & Anomaly Investigation

## Objective
This notebook performs a comprehensive data quality audit of the canonical IBM Telco Customer Churn dataset, covering:
1. Structural integrity, dimensions, and data types.
2. Missing value and whitespace string diagnostics.
3. Customer identifier uniqueness and duplicate analysis.
4. Ground-truth target distribution.
5. In-depth empirical investigation of the 11 whitespace-only `TotalCharges` values.

This audit utilizes PredictLab's reusable profiling module (`predictlab.profiling.quality`).


In [1]:
from predictlab.data.load import load_raw_dataset
from predictlab.profiling.quality import (
    detect_whitespace_entries,
    investigate_total_charges,
    profile_data_quality,
)
from predictlab.profiling.summary import (
    compute_dataset_overview,
    compute_target_distribution,
)

print("Modules imported successfully.")

Modules imported successfully.


## 1. Load Canonical Dataset & Structural Inspection

We load the dataset through the project's immutable loading interface (`load_raw_dataset`).

In [2]:
df = load_raw_dataset()
overview = compute_dataset_overview(df)
for k, v in overview.items():
    print(f"{k}: {v}")

df.info()

total_observations: 7043
total_features: 21
numeric_features_count: 3
categorical_features_count: 17
identifier_column: customerID
unique_customers: 7043
duplicate_rows: 0
duplicate_identifiers: 0
memory_usage_mb: 6.82
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport

## 2. Missing Value and Whitespace String Diagnostics

Standard pandas `.isna()` checks find zero missing values because whitespace strings (`" "`) are treated as valid object strings. We perform an explicit whitespace audit.

In [3]:
ws_results = detect_whitespace_entries(df)
print("Columns with whitespace-only entries:", list(ws_results.keys()))
for col, idxs in ws_results.items():
    print(f"Column '{col}' contains {len(idxs)} whitespace-only values at row indices: {idxs}")

Columns with whitespace-only entries: ['TotalCharges']
Column 'TotalCharges' contains 11 whitespace-only values at row indices: [488, 753, 936, 1082, 1340, 3331, 3826, 4380, 5218, 6670, 6754]


## 3. Duplicate Records and Identifier Integrity

We verify whether any complete duplicate records or duplicate customer identifiers exist.

In [4]:
dup_rows = int(df.duplicated().sum())
dup_ids = int(df['customerID'].duplicated().sum())
print(f"Complete duplicate rows: {dup_rows}")
print(f"Duplicate customerID values: {dup_ids}")
print(f"Total distinct customer IDs: {df['customerID'].nunique()} (out of {len(df)} rows)")

Complete duplicate rows: 0
Duplicate customerID values: 0
Total distinct customer IDs: 7043 (out of 7043 rows)


## 4. Target Class Distribution (`Churn`)

We inspect the class frequencies, percentages, and imbalance ratio.

In [5]:
target_info = compute_target_distribution(df)
for k, v in target_info.items():
    print(f"{k}: {v}")

target_column: Churn
total_records: 7043
positive_class: Yes
negative_class: No
positive_count: 1869
negative_count: 5174
positive_percentage: 26.54
negative_percentage: 73.46
negative_to_positive_ratio: 2.7683
positive_to_negative_ratio: 0.3612
class_balance_description: Moderate class imbalance (~2.77:1 retained vs churned accounts). Positive churn rate is 26.54%.


## 5. Mandatory Investigation: `TotalCharges` Whitespace Anomaly

We examine the 11 rows where `TotalCharges` contains whitespace strings and evaluate their relationship with `tenure`, `MonthlyCharges`, `Contract`, and `Churn`.

In [6]:
inv = investigate_total_charges(df)
print(f"Total whitespace rows: {inv.total_whitespace_rows}")
print(f"All zero tenure? {inv.all_zero_tenure}")
print(f"MonthlyCharges: min={inv.monthly_charges_min}, mean={inv.monthly_charges_mean:.2f}, max={inv.monthly_charges_max}")
print(f"Contract distribution: {inv.contract_distribution}")
print(f"Churn distribution: {inv.churn_distribution}")
print(f"Tenure==1 exact match ratio (TotalCharges == MonthlyCharges): {inv.tenure_one_exact_match_ratio:.4f}")
print(f"Spearman correlation between valid TotalCharges and tenure: {inv.tenure_totalcharges_spearman:.4f}")
print("\n--- RECOMMENDATION ---")
print(inv.recommendation)
print("\n--- RATIONALE ---")
print(inv.rationale)

Total whitespace rows: 11
All zero tenure? True
MonthlyCharges: min=19.7, mean=41.42, max=80.85
Contract distribution: {'Two year': 10, 'One year': 1}
Churn distribution: {'No': 11}
Tenure==1 exact match ratio (TotalCharges == MonthlyCharges): 1.0000
Spearman correlation between valid TotalCharges and tenure: 0.8892

--- RECOMMENDATION ---
Impute 0.0 in Phase 3 preprocessing pipeline.

--- RATIONALE ---
All 11 whitespace rows strictly coincide with tenure == 0 (all_zero_tenure=True). For tenure == 1, 100% of accounts (613) exhibit TotalCharges == MonthlyCharges (match ratio: 1.0000), demonstrating that TotalCharges represents accumulated completed billing cycles. At enrollment (tenure == 0), accumulated charges billed to date are mathematically and operationally $0.00. Setting TotalCharges = 0.0 maintains physical fidelity, avoids dropping valid subscriber contracts, and preserves all 7,043 observations without distortion.


## 6. Comprehensive Data Quality Report

We execute the full automated data quality profiling routine (`profile_data_quality`).

In [7]:
quality_report = profile_data_quality(df)
print("Quality Summary Notes:")
for note in quality_report.quality_summary_notes:
    print(f"  • {note}")

Quality Summary Notes:


  • Dataset contains 7043 records and 21 columns with 0 completely duplicated rows.
  • Identifier 'customerID' has 7043 distinct values (0 duplicates).
  • Detected whitespace-only strings in columns: ['TotalCharges']. TotalCharges contains 11 whitespace entries strictly at tenure==0.
